In [1]:
import optuna

# You can use Matplotlib instead of Plotly for visualization by simply replacing `optuna.visualization` with
# `optuna.visualization.matplotlib` in the following examples.
from optuna.visualization import plot_contour
from optuna.visualization import plot_edf
from optuna.visualization import plot_intermediate_values
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_param_importances
from optuna.visualization import plot_rank
from optuna.visualization import plot_slice
from optuna.visualization import plot_timeline

In [2]:
study_name = "hpo_multi_qrs_aligned2"
model_name = "BiXLSTM2new"
sqlite_db_path = f"results/{study_name}/{model_name}/db.sqlite3"


study = optuna.create_study(study_name=study_name, storage=f'sqlite:///{sqlite_db_path}', load_if_exists=True)

[I 2025-05-28 11:58:02,692] Using an existing study with name 'hpo_multi_qrs_aligned2' instead of creating a new one.


In [3]:
print(study.trials[0].user_attrs)
print((study.trials[0].params))

{'block_config': '2_ms', 'num_blocks_slstm_at': '2_[1]'}
{'num_blocks_slstm_at': 4, 'dropout': 0.25, 'embedding_dim': 32, 'conv1d_kernel_size': 51, 'num_heads': 2}


In [4]:
config_dict = {
        t.params["num_blocks_slstm_at"]: {
            "config_str": t.user_attrs["num_blocks_slstm_at"],
            "num_blocks": t.user_attrs["num_blocks_slstm_at"].split("_")[0],
            "slstm_at": t.user_attrs["num_blocks_slstm_at"].split("_")[1],
        } for t in study.trials
    }

def get_config(id):
    config_dict[id]["num_blocks"], config_dict[id]["slstm_at"]
def get_config_str(id):
    config_dict[id]["config_str"]

In [5]:
plot_optimization_history(study, target=lambda t: t.values[0], target_name="macro f1")

In [6]:
plot_optimization_history(study, target=lambda t: t.values[1], target_name="number of parameters")

In [7]:
plot_intermediate_values(study)

[W 2025-05-19 11:12:06,267] You need to set up the pruning feature to utilize `plot_intermediate_values()`


In [8]:
plot_slice(study, target = lambda t: t.values[0], target_name="macro f1")

In [9]:
plot_slice(study, target = lambda t: t.values[1], target_name="number of parameters")

In [10]:
fig = optuna.visualization.plot_pareto_front(study, targets=lambda t: (t.values[1], t.values[0]), target_names=["# Parameters", "macro f1"])
fig.show()

In [3]:
import plotly.graph_objects as go
from plotly.graph_objects import Contour, Figure
import pandas as pd
import os
import plotly.express as px



# Load csv into a pandas DataFrame




def plot_contour_plot(df, db_name="", fig_title="", x_title="macrof1_training", y_title=None, x=None, y=None, z=None, text=None):
    if y_title is None:
        y_title = f"tolerance-aware macrof1 ({db_name})"
    ax="macrof1_training"
    #ay="evaluation_metrics__tolerance_aware_classification_report__weighted avg__f1-score"
    ay="evaluation_metrics__tolerance_aware_classification_report__macro avg__f1-score"
    
    az="total_params"

    if x is None:
        x, y, z, text = [], [], [], []
        for i, row in df.iterrows():
            x.append((row[ax]))
            y.append((row[ay]))
            z.append((row[az]))
            text.append(str(row["config"]))


    fig = go.Figure()
    fig.add_trace(go.Contour(
        x=x,
        y=y,
        z=z,
        line_smoothing=0,
        hovertemplate=(
                '<extra></extra>'    # This removes the trace name from the hover label
            ),
    ))
    fig.update_layout(autosize=True)
    fig.add_trace(go.Scatter(
        x=x, 
        y=y, 
        text=[f"params: {z[i]}<br>Run: {text[i]}" for i in range(len(z))],
        mode='markers', 
        marker=dict(size=2), 
        hovertemplate=(
                'X: %{x}<br>'       # Display X-coordinate
                'Y: %{y}<br>'       # Display Y-coordinate
                '%{text}<br>'  # Display the config string
                '<extra></extra>'    # This removes the trace name from the hover label
            ),
    ))

    fig.update_layout(
        title=fig_title,
        xaxis_title=x_title,
        yaxis_title=y_title,
        height=1000,
        width=1000,
    )

    fig.show()
    
    
def plot_scatter_plot(df, db_name="",fig_title="", x_title="num_params", y_title=None, x=None, y=None, text=None):
    if y_title is None:
        y_title = f"tolerance-aware macrof1 ({db_name})"
    ax="total_params"
    ay="evaluation_metrics__tolerance_aware_classification_report__macro avg__f1-score"

    if x is None:
        x, y, text = [], [], []
        for i, row in df.iterrows():
            x.append((row[ax]))
            y.append((row[ay]))
            text.append(str(row["config"]))

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x, 
        y=y, 
        text=[f"Run: {text[i]}" for i in range(len(x))],
        mode='markers', 
        marker=dict(size=5), 
        hovertemplate=(
                'X: %{x}<br>'       # Display X-coordinate
                'Y: %{y}<br>'       # Display Y-coordinate
                '%{text}<br>'  # Display the config string
                '<extra></extra>'    # This removes the trace name from the hover label
            ),
    ))

    fig.update_layout(
        title=fig_title,
        xaxis_title=x_title,
        yaxis_title=y_title,
        height=600,
        width=1000,
    )

    fig.show()
    



In [62]:
plot_contour_plot(pd.read_csv(os.path.join("results", study_name, model_name, "ludb_merged_aligned.csv"), delimiter=";"), db_name="ludb")
plot_scatter_plot(pd.read_csv(os.path.join("results", study_name, model_name, "ludb_merged_aligned.csv"), delimiter=";"), db_name="ludb")

In [60]:
plot_contour_plot(pd.read_csv(os.path.join("results", study_name, model_name, "ludb_filtered_merged_aligned.csv"), delimiter=";"), db_name="ludb_filtered")
plot_scatter_plot(pd.read_csv(os.path.join("results", study_name, model_name, "ludb_filtered_merged_aligned.csv"), delimiter=";"), db_name="ludb_filtered")

In [ ]:
plot_contour_plot(pd.read_csv(os.path.join("results", study_name, model_name, "ludb_filtered_merged_aligned.csv"), delimiter=";"), db_name="ludb_filtered")
plot_scatter_plot(pd.read_csv(os.path.join("results", study_name, model_name, "ludb_filtered_merged_aligned.csv"), delimiter=";"), db_name="ludb_filtered")

In [37]:
plot_contour_plot(pd.read_csv(os.path.join("results", study_name, model_name, "ludb_new_merged_aligned.csv"), delimiter=";"), db_name="ludb_new")
plot_scatter_plot(pd.read_csv(os.path.join("results", study_name, model_name, "ludb_new_merged_aligned.csv"), delimiter=";"), db_name="ludb_new")

In [53]:
plot_contour_plot(pd.read_csv(os.path.join("results", study_name, model_name, "lvad_merged_aligned.csv"), delimiter=";"), db_name="LVAD")
plot_scatter_plot(pd.read_csv(os.path.join("results", study_name, model_name, "lvad_merged_aligned.csv"), delimiter=";"), db_name="LVAD")

In [39]:
plot_contour_plot(pd.read_csv(os.path.join("results", study_name, model_name, "lvad_ecgs_new_merged_aligned.csv"), delimiter=";"), db_name="lvad_ecgs_new")
plot_scatter_plot(pd.read_csv(os.path.join("results", study_name, model_name, "lvad_ecgs_new_merged_aligned.csv"), delimiter=";"), db_name="lvad_ecgs_new")

In [40]:
plot_contour_plot(pd.read_csv(os.path.join("results", study_name, model_name, "lvad_ecgs_new_merged_aligned.csv"), delimiter=";"), db_name="lvad_ecgs_new")
plot_scatter_plot(pd.read_csv(os.path.join("results", study_name, model_name, "lvad_ecgs_new_merged_aligned.csv"), delimiter=";"), db_name="lvad_ecgs_new")

In [61]:
plot_contour_plot(pd.read_csv(os.path.join("results", study_name, model_name, "lvad_ecgs_new_merged_aligned.csv"), delimiter=";"), db_name="lvad_ecgs_new")
plot_scatter_plot(pd.read_csv(os.path.join("results", study_name, model_name, "lvad_ecgs_new_merged_aligned.csv"), delimiter=";"), db_name="lvad_ecgs_new")

In [31]:
plot_contour_plot(pd.read_csv(os.path.join("results", study_name, "BiXLSTM2new", "rabbit_merged_aligned.csv"), delimiter=";"), title="rabbit")
plot_scatter_plot(pd.read_csv(os.path.join("results", study_name, "BiXLSTM2new", "rabbit_merged_aligned.csv"), delimiter=";"), title="rabbit")


In [15]:
plot_param_importances(study)

In [16]:
plot_parallel_coordinate(study, target=lambda t: t.values[0], target_name="macro f1")

In [17]:
plot_parallel_coordinate(study, target=lambda t: t.values[1], target_name="number of parameters")

In [18]:
import plotly.express as px
import pandas as pd
from collections import OrderedDict
import random

dd = OrderedDict({})

#print(px.data.iris())

valid_df = pd.read_csv(os.path.join("results", study_name, "BiXLSTM2new", "ludb_merged_aligned.csv"), delimiter=";")
# find row in dataframe where a column is equal to a value"



for trial in study.trials:
    #print(valid_df["evaluation_metrics__tolerance_aware_classification_report__weighted avg__f1-score"].loc[trial.number].item())
    if trial.state == optuna.trial.TrialState.COMPLETE:
        dd[trial.number] = OrderedDict({
            "num_blocks": int(trial.user_attrs["num_blocks_slstm_at"].split("_")[0]),
            "slstm_at": trial.user_attrs["num_blocks_slstm_at"].split("_")[1],
            "config_id": trial.params["num_blocks_slstm_at"],
            "dropout": trial.params["dropout"],
            "embedding_dim": trial.params["embedding_dim"],
            "conv1d_kernel_size": trial.params["conv1d_kernel_size"],
            "macro_f1": trial.values[0],
            "valid_macro_f1": valid_df["evaluation_metrics__tolerance_aware_classification_report__weighted avg__f1-score"].loc[trial.number].item(),
            "total_params": trial.values[1],
            #"macro_f1": 0.993,
            #"total_params": 43556,
            "num_heads": trial.params["num_heads"],
        })


column_names = list(dd[0].keys())

d = {
    i: [dd[i][k] for k in column_names] for i in range(len(dd))
}

df = pd.DataFrame.from_dict(d, orient='index', columns=column_names)

print((df.loc[0:3]))

#color = "total_params"
color = "macro_f1"

#color_midpoint = 0.5 if color == "macro_f1" else max(df["total_params"])/2
#color_midpoint = max(df["macro_f1"])/2 if color == "macro_f1" else max(df["total_params"])/2

range_color = [min(df["macro_f1"]), max(df["macro_f1"])] if color == "macro_f1" else [min(df["total_params"]), max(df["total_params"])]

fig = px.parallel_coordinates(
    df, 
    color=color, 
    labels={
        "num_blocks": "xlstm blocks", 
        #"slstm_at": "slstm at", 
        "dropout": "Dropout",
        "conv1d_kernel_size": "Conv1D Kernel Size",
        "embedding_dim": "Embedding dimension", 
        "macro_f1": "Macro F1 Score",
        "valid_macro_f1": "tolerance-aware Macro F1 Score",
        "total_params": "Number of Parameters",
        "num_heads": "Number of Heads", 
    },
    dimensions=[
        "num_blocks", 
        #"slstm_at", 
        "dropout", 
        "embedding_dim", 
        "conv1d_kernel_size", 
        "macro_f1",
        "valid_macro_f1",
        "total_params",
        "num_heads", 
    ],
    color_continuous_scale=px.colors.diverging.Tealrose,
    color_continuous_midpoint=(range_color[1]-range_color[0])/2,
    range_color=range_color,)

# fig = px.parallel_coordinates(
#     df, 
#     color="species_id", 
#     labels={"species_id": "Species","sepal_width": "Sepal Width", "sepal_length": "Sepal Length", "petal_width": "Petal Width", "petal_length": "Petal Length", },
#     color_continuous_scale=px.colors.diverging.Tealrose,
#     color_continuous_midpoint=2)

# Hide the color scale that is useless in this case
fig.update_layout(coloraxis_showscale=False)

# Show the plot
fig.show()

   num_blocks slstm_at  config_id  dropout  embedding_dim  conv1d_kernel_size  \
0           2      [1]          4     0.25             32                  51   
1           3    [0,1]         10     0.20             16                  21   
2           3       []          6     0.15             12                  31   
3           3  [0,1,2]         13     0.15             36                  41   

   macro_f1  valid_macro_f1  total_params  num_heads  
0  0.899478        0.924742       21064.0          2  
1  0.886374        0.927088        7976.0          2  
2  0.883704        0.922131        6082.0          1  
3  0.891007        0.930831       36148.0          2  


In [19]:
import plotly.graph_objects as go


fig = go.Figure(data=
    go.Parcoords(
     #    line = dict(color = df['macro_f1'],
     #               colorscale = [[0,'purple'],[0.5,'lightseagreen'],[1,'gold']],
     #               showscale = True,
     #               cmin = df['macro_f1'].min(),
     #               cmax = df['macro_f1'].max(),
     #    ),
        line = dict(color = df['total_params'],
                   colorscale = [[0,'purple'],[0.5,'lightseagreen'],[1,'gold']],
                   showscale = True,
                   cmin = df['total_params'].min(),
                   cmax = df['total_params'].max(),
        ),
        dimensions = list([
            dict(range = [df['num_blocks'].min(),df['num_blocks'].max()],
                 label = "num blocks", values = df['num_blocks']),
            dict(range = [df['config_id'].min(),df['config_id'].max()],
                 tickvals = sorted(df['config_id']),
                 label = 'slstm at', values = df['config_id'],
                 ticktext = [config_dict[id]["slstm_at"] for id in sorted(df['config_id'])]),
            dict(range = [df['dropout'].min(),df['dropout'].max()],
                 label = "Dropout", values = df['dropout']),
            dict(range = [df['embedding_dim'].min(),df['embedding_dim'].max()],
                 label = "embedding_dim", values = df['embedding_dim']),
            dict(range = [df['conv1d_kernel_size'].min(),df['conv1d_kernel_size'].max()],
                 label = "conv1d_kernel_size", values = df['conv1d_kernel_size']),
            dict(range = [df['macro_f1'].min(),df['macro_f1'].max()],
                 label = "macro_f1", values = df['macro_f1']),
            dict(range = [df['total_params'].min(),df['total_params'].max()],
                 label = "total_params", values = df['total_params']),
            dict(range = [df['num_heads'].min(),df['num_heads'].max()],
                 label = "num_heads", values = df['num_heads']),
            ]
          )
    )
)
fig.show()


In [20]:
plot_contour(study, target=lambda t: t.values[0], target_name="macro f1")

In [21]:
plot_contour(study, target=lambda t: t.values[1], target_name="number of parameters")

In [22]:
plot_rank(study, target = lambda t: t.values[0], target_name="macro f1")


In [23]:
plot_rank(study, target = lambda t: t.values[1], target_name="number of parameters")

In [24]:
plot_timeline(study)

In [38]:
print(len(study.trials))
for t in study.trials:
    print(f"state: {t.state} number: {t.number}")

60
state: 1 number: 0
state: 1 number: 1
state: 1 number: 2
state: 1 number: 3
state: 1 number: 4
state: 1 number: 5
state: 1 number: 6
state: 1 number: 7
state: 1 number: 8
state: 1 number: 9
state: 1 number: 10
state: 1 number: 11
state: 1 number: 12
state: 1 number: 13
state: 1 number: 14
state: 1 number: 15
state: 1 number: 16
state: 1 number: 17
state: 1 number: 18
state: 1 number: 19
state: 1 number: 20
state: 1 number: 21
state: 1 number: 22
state: 1 number: 23
state: 1 number: 24
state: 1 number: 25
state: 1 number: 26
state: 1 number: 27
state: 1 number: 28
state: 1 number: 29
state: 1 number: 30
state: 1 number: 31
state: 1 number: 32
state: 1 number: 33
state: 1 number: 34
state: 1 number: 35
state: 1 number: 36
state: 1 number: 37
state: 1 number: 38
state: 1 number: 39
state: 1 number: 40
state: 1 number: 41
state: 1 number: 42
state: 1 number: 43
state: 1 number: 44
state: 1 number: 45
state: 1 number: 46
state: 1 number: 47
state: 1 number: 48
state: 1 number: 49
state: 

In [64]:
df_ludb_filtered = pd.read_csv(os.path.join("results", study_name, model_name, "ludb_filtered_merged_aligned.csv"), delimiter=";")
df_lvad_ecgs_new = pd.read_csv(os.path.join("results", study_name, model_name, "lvad_ecgs_new_merged_aligned.csv"), delimiter=";")

x = df_ludb_filtered["evaluation_metrics__tolerance_aware_classification_report__macro avg__f1-score"].values
y = df_lvad_ecgs_new["evaluation_metrics__tolerance_aware_classification_report__macro avg__f1-score"].values
z = df_ludb_filtered["total_params"].values
text = df_ludb_filtered["config"].values

plot_contour_plot(None, x=x, y=y, z=z, text=text, fig_title="ludb_filtered vs lvad_ecgs_new", x_title="ludb_filtered macro f1", y_title="lvad_ecgs_new macro f1")
plot_scatter_plot(None, x=x, y=y, text=text, fig_title="ludb_filtered vs lvad_ecgs_new", x_title="ludb_filtered macro f1", y_title="lvad_ecgs_new macro f1")

In [66]:

df_qtdb = pd.read_csv(os.path.join("results", "hpo_multi_qrs_aligned2_ludb", "BiXLSTM2new", "qtdb_merged_aligned.csv"), delimiter=";")
df_lvad_ecgs_new = pd.read_csv(os.path.join("results", "hpo_multi_qrs_aligned2_ludb", "BiXLSTM2new", "lvad_ecgs_new_merged_aligned.csv"), delimiter=";")

x = df_qtdb["evaluation_metrics__tolerance_aware_classification_report__macro avg__f1-score"].values
y = df_lvad_ecgs_new["evaluation_metrics__tolerance_aware_classification_report__macro avg__f1-score"].values
z = df_qtdb["total_params"].values
text = df_qtdb["config"].values

plot_contour_plot(None, x=x, y=y, z=z, text=text, fig_title="qtdb vs lvad_ecgs_new (trained with ludb)", x_title="qtdb macro f1", y_title="lvad_ecgs_new macro f1")
plot_scatter_plot(None, x=x, y=y, text=text, fig_title="qtdb vs lvad_ecgs_new (trained with ludb)", x_title="qtdb macro f1", y_title="lvad_ecgs_new macro f1")

In [14]:
def plot_contour_plot_3_f1(x, y, z, fig_title="", x_title="x", y_title="y", z_title="z"):
    fig = go.Figure()
    fig.add_trace(go.Contour(
        x=x,
        y=y,
        z=z,
        line_smoothing=0,
        hovertemplate=(
                '<extra></extra>'    # This removes the trace name from the hover label
            ),
    ))
    fig.update_layout(autosize=True)
    fig.add_trace(go.Scatter(
        x=x, 
        y=y, 
        text=text,
        mode='markers', 
        marker=dict(size=4, color="black"), 
        hovertemplate=(
                #'X: %{x}<br>'       # Display X-coordinate
                #'Y: %{y}<br>'       # Display Y-coordinate
                '%{text}<br>'  # Display the config string
                '<extra></extra>'    # This removes the trace name from the hover label
            ),
    ))

    fig.update_layout(
        title=fig_title,
        xaxis_title=x_title,
        yaxis_title=y_title,
        height=1000,
        width=1000,
    )

    fig.show()
    

In [15]:
df_ludb = pd.read_csv(os.path.join("results", "hpo_multi_qrs_aligned2", "BiXLSTM2new", "ludb_qrs_aligned_padded_merged_aligned.csv"), delimiter=";")
df_lvad = pd.read_csv(os.path.join("results", "hpo_multi_qrs_aligned2", "BiXLSTM2new", "lvad_ecgs_new_qrs_aligned_padded_merged_aligned.csv"), delimiter=";")
df_rabbit = pd.read_csv(os.path.join("results", "hpo_multi_qrs_aligned2", "BiXLSTM2new", "rabbit_qrs_aligned_padded_merged_aligned.csv"), delimiter=";")
#df_rabbit = pd.read_csv(os.path.join("results", "hpo_multi_qrs_aligned2", "BiXLSTM2new", "rabbit_merged_aligned.csv"), delimiter=";")

x = df_ludb["evaluation_metrics__tolerance_aware_classification_report__macro avg__f1-score"].values
y = df_lvad["evaluation_metrics__tolerance_aware_classification_report__macro avg__f1-score"].values
z = df_rabbit["evaluation_metrics__tolerance_aware_classification_report__macro avg__f1-score"].values

params = df_ludb["total_params"].values
configs = df_ludb["config"].values
text = [f"ludb f1: {x[i]}<br>lvad f1: {y[i]}<br>rabbit f1: {z[i]}<br>config: {configs[i]}<br>params: {params[i]}" for i in range(len(z))]



plot_contour_plot_3_f1(x, y, z, fig_title="ludb, lvad, rabbit", x_title="ludb macro f1", y_title="lvad macro f1", z_title="rabbit macro f1")

In [18]:
df_ludb = pd.read_csv(os.path.join("results", "hpo_multi_qrs_aligned2", "BiXLSTM2new", "ludb_qrs_aligned_padded_merged_aligned.csv"), delimiter=";")
df_lvad = pd.read_csv(os.path.join("results", "hpo_multi_qrs_aligned2", "BiXLSTM2new", "lvad_ecgs_new_qrs_aligned_padded_merged_aligned.csv"), delimiter=";")
df_rabbit = pd.read_csv(os.path.join("results", "hpo_multi_qrs_aligned2", "BiXLSTM2new", "rabbit_qrs_aligned_padded_merged_aligned.csv"), delimiter=";")

x = df_ludb["evaluation_metrics__tolerance_aware_classification_report__macro avg__f1-score"].values
y = df_lvad["evaluation_metrics__tolerance_aware_classification_report__macro avg__f1-score"].values
rabbit_f1 = df_rabbit["evaluation_metrics__tolerance_aware_classification_report__macro avg__f1-score"].values
z = df_ludb["total_params"].values

params = df_ludb["total_params"].values
configs = df_ludb["config"].values
text = [f"ludb f1: {x[i]}<br>lvad f1: {y[i]}<br>params: {z[i]}<br>rabbit f1: {rabbit_f1[i]}<br>config: {configs[i]}" for i in range(len(z))]



plot_contour_plot_3_f1(x, y, z, fig_title="ludb, lvad, params", x_title="ludb macro f1", y_title="lvad macro f1", z_title="params")